# 04 — Escalar y revalidar

Los pesos de la fase 3 salieron de **40 pedidos cortos**. Aquí:

1. Muestra **aleatoria** del split completo (80 / 20 / 20), semilla 42. Los 5 demo siguen fuera.
2. Colecta P2O + reentrena linear y MLP (mismo encoder, mismo contrato).
3. Compara en val/test **scale** contra heurístico, placeholder y los pesos de la fase 3.
4. Si el MLP `p=1,s=1` no empeora utilización (y todo es válido), pasa a ser el `model_path` de producción.

No pisa `data/train|val|test` de las fases 1–3. Todo nuevo va a `data/scale/`.

In [2]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")

REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

print("ML_ROOT =", ML_ROOT)

ML_ROOT = /home/edmundo/packing-services/online_policy_ml


In [3]:
from config import HEURISTIC_P3S2_MAX_ORDERS, HIDDEN_SIZE, MLP_EPOCHS, REGIMES, SCALE_COUNTS, SEED, TEACHER_NAME, mlp_lr
from paths import (
    BED_JSON_NAME,
    HOLDOUT_DIR,
    MODELS_DIR,
    ORDER_IDS_NAME,
    PLACEHOLDER_LINEAR,
    REPORTS_DIR,
    SOURCE_BED_BPP,
    SPLITS_DIR,
    model_dir_fase,
    model_path_linear,
    model_path_mlp,
    scale_split_dir,
    scale_transitions_path,
)
from splits import (
    load_orders,
    materialize_scale,
    order_item_count,
    order_target,
    scale_subset,
    summarize_orders,
)
from collect import collect_dataset, load_transitions, save_transitions
from train_linear import fit_linear, linear_accuracy
from train_mlp import fit_mlp
from export_ckpt import export_linear_json, export_mlp_pt
from evaluate import evaluate_orders, smoke_from_path, write_report
from packing_services.online.features import FEATURE_DIM, FEATURE_NAMES
from packing_services.online.learned.policy import LearnedPlacementPolicy
from packing_services.datasets.bed_bpp import smallest_order_id

assert (SPLITS_DIR / "full_split.json").is_file(), "Falta fase 1 (full_split.json)"
assert model_path_linear().is_file(), "Falta fase 3 (linear_v1.json)"
assert PLACEHOLDER_LINEAR.is_file()

full_split = json.loads((SPLITS_DIR / "full_split.json").read_text(encoding="utf-8"))
working = json.loads((SPLITS_DIR / "working_split.json").read_text(encoding="utf-8"))
blocked = set(json.loads((SPLITS_DIR / "blocked_demo_ids.json").read_text(encoding="utf-8"))["order_ids"])
print("SCALE_COUNTS", SCALE_COUNTS, "semilla", SEED)

SCALE_COUNTS {'train': 80, 'val': 20, 'test': 20} semilla 42


## Muestra scale (aleatoria, no la más corta)

In [5]:
source_orders = load_orders(SOURCE_BED_BPP)
scale = scale_subset(full_split)

for name, ids in scale.items():
    leak = set(ids) & blocked
    assert not leak, f"fuga demo en scale/{name}: {leak}"
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (set(scale[a]) & set(scale[b]))

written = materialize_scale(source_orders, scale)
rows = []
for name in ("train", "val", "test"):
    w_ids = working[name]
    s_ids = scale[name]
    rows.append({
        "split": name,
        "working_n": len(w_ids),
        "working_items_mean": round(sum(order_item_count(source_orders[i]) for i in w_ids) / len(w_ids), 1),
        "scale_n": len(s_ids),
        "scale_items_mean": round(sum(order_item_count(source_orders[i]) for i in s_ids) / len(s_ids), 1),
        "scale_items_min": min(order_item_count(source_orders[i]) for i in s_ids),
        "scale_items_max": max(order_item_count(source_orders[i]) for i in s_ids),
        "overlap_with_working": len(set(s_ids) & set(w_ids)),
    })
display(pd.DataFrame(rows))
print("archivos:", {k: str(v) for k, v in written.items()})

   split  working_n  ...  scale_items_max  overlap_with_working
0  train         24  ...               91                     0
1    val          8  ...               94                     1
2   test          8  ...               63                     0

[3 rows x 8 columns]
archivos: {'train': '/home/edmundo/packing-services/online_policy_ml/data/scale/train/bed_bpp_orders.json', 'val': '/home/edmundo/packing-services/online_policy_ml/data/scale/val/bed_bpp_orders.json', 'test': '/home/edmundo/packing-services/online_policy_ml/data/scale/test/bed_bpp_orders.json'}


## Colecta P2O sobre la muestra scale

In [7]:
datasets = {name: load_orders(scale_split_dir(name) / BED_JSON_NAME) for name in ("train", "val", "test")}
ids = {name: json.loads((scale_split_dir(name) / ORDER_IDS_NAME).read_text(encoding="utf-8")) for name in ("train", "val", "test")}

reports = []
for lookahead_p, select_s in REGIMES:
    print(f"\n=== p={lookahead_p} s={select_s} ===")
    for split_name in ("train", "val", "test"):
        path = scale_transitions_path(split_name, lookahead_p, select_s)
        if path.is_file():
            payload = load_transitions(path)
            print(f"  {split_name}: reusa {path.name}")
        else:
            payload = collect_dataset(
                datasets[split_name],
                ids[split_name],
                lookahead_p=lookahead_p,
                select_s=select_s,
                teacher=TEACHER_NAME,
            )
            save_transitions(payload, path)
        n_opt = [t["n_options"] for t in payload["transitions"]]
        row = {
            "split": split_name,
            "p": lookahead_p,
            "s": select_s,
            "n_orders": len(ids[split_name]),
            "n_transitions": payload["n_transitions"],
            "label_rate": payload["label_rate"],
            "n_options_mean": round(sum(n_opt) / max(len(n_opt), 1), 2),
            "seconds": round(float(payload.get("seconds") or 0.0), 2),
        }
        reports.append(row)
        print(f"  {split_name}: {row['n_transitions']} tr, label_rate={row['label_rate']:.2f}")

df_tr = pd.DataFrame(reports)
display(df_tr)
assert df_tr["label_rate"].min() == 1.0



=== p=1 s=1 ===
  train: reusa transitions_p1s1.pkl
  train: 3207 tr, label_rate=1.00
  val: reusa transitions_p1s1.pkl
  val: 832 tr, label_rate=1.00
  test: reusa transitions_p1s1.pkl
  test: 776 tr, label_rate=1.00

=== p=3 s=2 ===
  train: reusa transitions_p3s2.pkl
  train: 3230 tr, label_rate=1.00
  val: reusa transitions_p3s2.pkl
  val: 844 tr, label_rate=1.00
  test: reusa transitions_p3s2.pkl
  test: 776 tr, label_rate=1.00
   split  p  s  n_orders  n_transitions  label_rate  n_options_mean  seconds
0  train  1  1        80           3207         1.0           38.27   147.21
1    val  1  1        20            832         1.0           41.48    47.92
2   test  1  1        20            776         1.0           42.89    37.24
3  train  3  2        80           3230         1.0           85.87   371.20
4    val  3  2        20            844         1.0           96.00   145.83
5   test  3  2        20            776         1.0           89.34    81.18


## Reentrenar (se guarda en `artifacts/models/fase4/`; la fase 3 se copia a `fase3/`)

In [9]:
fase3_dir = model_dir_fase("fase3")
fase4_dir = model_dir_fase("fase4")
for src in (model_path_linear(), model_path_mlp(1, 1), model_path_mlp(3, 2)):
    if src.is_file():
        dst = fase3_dir / src.name
        if not dst.is_file():
            shutil.copy2(src, dst)
            print("backup fase3", dst)
        else:
            print("fase3 ya tenía", dst.name)

train_p1 = load_transitions(scale_transitions_path("train", 1, 1))
val_p1 = load_transitions(scale_transitions_path("val", 1, 1))
train_p3 = load_transitions(scale_transitions_path("train", 3, 2))
val_p3 = load_transitions(scale_transitions_path("val", 3, 2))

lin = fit_linear(train_p1["transitions"], epochs=8, lr=3e-4, l2=1e-5, seed=SEED)
acc_tr = linear_accuracy(train_p1["transitions"], lin["weights"], lin["bias"])
acc_va = linear_accuracy(val_p1["transitions"], lin["weights"], lin["bias"])
linear_f4 = fase4_dir / "linear_v1.json"
export_linear_json(
    lin["weights"], lin["bias"], linear_f4,
    notes=f"fase4 BC P2O {TEACHER_NAME} seed={SEED} scale {SCALE_COUNTS}",
)
print("linear acc train/val", round(acc_tr, 3), round(acc_va, 3), "→", linear_f4)

mlp_f4 = {}
mlp_histories = {}
for (p, s), tr, va in (((1, 1), train_p1, val_p1), ((3, 2), train_p3, val_p3)):
    print(f"\n--- MLP p={p} s={s} ---")
    fit = fit_mlp(
        tr["transitions"], va["transitions"],
        hidden_size=HIDDEN_SIZE, epochs=MLP_EPOCHS, lr=mlp_lr(p, s), seed=SEED,
    )
    path = fase4_dir / f"mlp_v1_p{p}s{s}.pt"
    export_mlp_pt(fit["state_dict"], path, hidden_size=HIDDEN_SIZE)
    mlp_f4[(p, s)] = path
    mlp_histories[(p, s)] = fit["history"]
    print(f"  lr={mlp_lr(p, s)} best_epoch={fit['best_epoch']} val_acc={fit['best_val_acc']:.4f}")
    print(pd.DataFrame(fit["history"]).round(4).tail(2).to_string(index=False))
    print("export", path)

pd.DataFrame(lin["history"]).plot(x="epoch", y=["loss", "accuracy"], figsize=(5.5, 3), title="Linear scale")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, key in zip(axes, ((1, 1), (3, 2))):
    hist = pd.DataFrame(mlp_histories[key])
    ax.plot(hist["epoch"], hist["train_acc"], label="train acc")
    ax.plot(hist["epoch"], hist["val_acc"], label="val acc")
    ax.set_title(f"MLP p={key[0]} s={key[1]}")
    ax.legend()
fig.tight_layout(); plt.show()

fase3 ya tenía linear_v1.json
fase3 ya tenía mlp_v1_p1s1.pt
fase3 ya tenía mlp_v1_p3s2.pt
linear acc train/val 0.933 0.946 → /home/edmundo/packing-services/online_policy_ml/artifacts/models/fase4/linear_v1.json

--- MLP p=1 s=1 ---
 epoch  train_loss  train_acc  val_loss  val_acc
   9.0      0.8754     0.9364    0.9705   0.9603
  10.0      1.0871     0.9348    0.3475   0.9627
export /home/edmundo/packing-services/online_policy_ml/artifacts/models/fase4/mlp_v1_p1s1.pt

--- MLP p=3 s=2 ---
 epoch  train_loss  train_acc  val_loss  val_acc
   9.0    191.5342     0.6861  198.5740   0.8009
  10.0    191.1197     0.6913   89.5234   0.7571
export /home/edmundo/packing-services/online_policy_ml/artifacts/models/fase4/mlp_v1_p3s2.pt


04_escalar_revalidar.ipynb:cell8:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
04_escalar_revalidar.ipynb:cell8:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Validación en val/test scale (misma muestra, pedidos más representativos)

In [11]:
orders_val = datasets["val"]
orders_test = datasets["test"]
holdout = load_orders(HOLDOUT_DIR / "5_bed-bpp.json")
fase3_linear = fase3_dir / "linear_v1.json"
fase3_mlp = fase3_dir / "mlp_v1_p1s1.pt"
fase3_mlp_p3 = fase3_dir / "mlp_v1_p3s2.pt"

def pack_table(orders, order_ids, lookahead_p, select_s, max_orders=None, extra=None, skip_heuristic=False):
    rows = []
    if not skip_heuristic:
        rows += evaluate_orders(orders, order_ids, engine="heuristic", lookahead_p=lookahead_p, select_s=select_s, max_orders=max_orders)
    rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(PLACEHOLDER_LINEAR), max_orders=max_orders)
    rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(linear_f4), max_orders=max_orders)
    mlp_now = mlp_f4.get((lookahead_p, select_s))
    if mlp_now:
        rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(mlp_now), max_orders=max_orders)
    if extra:
        for label, path in extra:
            if path.is_file():
                rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(path), max_orders=max_orders)
    return rows

def pretty(rows):
    df = pd.DataFrame(rows)
    df["model"] = df.apply(
        lambda r: r["engine"] if r["engine"] == "heuristic" else Path(str(r["model_path"])).name
        + ("#fase3" if "fase3" in str(r["model_path"]) else "")
        + ("#fase4" if "fase4" in str(r["model_path"]) else ""),
        axis=1,
    )
    return df.groupby("model", as_index=False).agg(
        n=("order_id", "count"),
        util=("volume_utilization", "mean"),
        packed=("items_packed", "mean"),
        valid=("is_valid", "mean"),
        sec=("seconds", "mean"),
    ).round(4)

print("val scale p=1 s=1 ...")
rows_val_p1 = pack_table(
    orders_val, ids["val"], 1, 1,
    extra=[("fase3_linear", fase3_linear), ("fase3_mlp", fase3_mlp)],
)
print("test scale p=1 s=1 ...")
rows_test_p1 = pack_table(
    orders_test, ids["test"], 1, 1,
    extra=[("fase3_linear", fase3_linear), ("fase3_mlp", fase3_mlp)],
)
print("val scale p=3 s=2 (sin heurístico: lookahead demasiado caro en pedidos medianos)")
rows_val_p3 = pack_table(
    orders_val, ids["val"], 3, 2,
    extra=[("fase3_mlp_p3", fase3_mlp_p3)],
    skip_heuristic=True,
)

print("\nVAL scale p=1 s=1"); display(pretty(rows_val_p1))
print("TEST scale p=1 s=1"); display(pretty(rows_test_p1))
print("VAL scale p=3 s=2"); display(pretty(rows_val_p3))
assert pretty(rows_val_p1)["valid"].min() == 1.0
assert pretty(rows_test_p1)["valid"].min() == 1.0


val scale p=1 s=1 ...
test scale p=1 s=1 ...
val scale p=3 s=2 (sin heurístico: lookahead demasiado caro en pedidos medianos)

VAL scale p=1 s=1
                          model   n    util  packed  valid     sec
0                     heuristic  20  0.6695   41.60    1.0  2.4022
1          linear_v1.json#fase3  20  0.6736   41.90    1.0  2.5396
2          linear_v1.json#fase4  20  0.6723   41.85    1.0  2.4837
3          mlp_v1_p1s1.pt#fase3  20  0.6753   41.95    1.0  2.5295
4          mlp_v1_p1s1.pt#fase4  20  0.6726   41.90    1.0  2.5611
5  online_policy_linear_v1.json  20  0.6749   41.80    1.0  2.4317
TEST scale p=1 s=1
                          model   n    util  packed  valid     sec
0                     heuristic  20  0.6716   38.80    1.0  1.8560
1          linear_v1.json#fase3  20  0.6698   38.60    1.0  1.8755
2          linear_v1.json#fase4  20  0.6769   39.10    1.0  1.8498
3          mlp_v1_p1s1.pt#fase3  20  0.6721   38.70    1.0  1.8681
4          mlp_v1_p1s1.pt#fase4 

## Promoción a producción y smoke de los 5 demo

In [13]:
def util_of(rows, needle):
    g = pretty(rows)
    hit = g[g["model"].str.contains(needle, regex=False)]
    if hit.empty:
        return None
    return float(hit.iloc[0]["util"])

u_f4 = util_of(rows_val_p1, "mlp_v1_p1s1.pt#fase4")
u_f3 = util_of(rows_val_p1, "mlp_v1_p1s1.pt#fase3")
u_heu = util_of(rows_val_p1, "heuristic")
print("val util MLP p1s1 fase4 / fase3 / heurístico:", u_f4, u_f3, u_heu)

promote = u_f4 is not None and (u_f3 is None or u_f4 + 1e-9 >= u_f3)
if promote:
    shutil.copy2(linear_f4, model_path_linear())
    shutil.copy2(mlp_f4[(1, 1)], model_path_mlp(1, 1))
    print("PROMOCIÓN p=1 s=1: linear + mlp_v1_p1s1.pt. No se toca mlp_v1_p3s2.pt")
else:
    print("SIN PROMOCIÓN: se mantienen los pesos de producción. Fase 4 queda en artifacts/models/fase4/")

prod_mlp = model_path_mlp(1, 1)
prod_linear = model_path_linear()
for p in (prod_linear, prod_mlp, model_path_mlp(3, 2)):
    LearnedPlacementPolicy.from_path(p)
    print("carga OK", p.name)

smoke_id = smallest_order_id(holdout)
smoke_rows = []
for name, path in (("linear_prod", prod_linear), ("mlp_p1s1_prod", prod_mlp)):
    row = smoke_from_path(path, holdout, smoke_id)
    row["checkpoint"] = name
    smoke_rows.append(row)
    print(name, "valid", row["is_valid"], "packed", row["items_packed"], "util", round(row["volume_utilization"], 4))
    assert row["is_valid"]

display(pd.DataFrame(smoke_rows)[["checkpoint", "order_id", "volume_utilization", "items_packed", "is_valid"]])

val util MLP p1s1 fase4 / fase3 / heurístico: 0.6726 0.6753 0.6695
SIN PROMOCIÓN: se mantienen los pesos de producción (fase 3). Fase 4 queda en artifacts/models/fase4/
carga OK linear_v1.json
carga OK mlp_v1_p1s1.pt
carga OK mlp_v1_p3s2.pt
linear_prod valid True packed 26 util 0.6464
mlp_p1s1_prod valid True packed 26 util 0.6464
      checkpoint  order_id  volume_utilization  items_packed  is_valid
0    linear_prod  00100408            0.646376            26      True
1  mlp_p1s1_prod  00100408            0.646376            26      True


In [14]:
report = {
    "fase": 4,
    "seed": SEED,
    "hidden_size": HIDDEN_SIZE,
    "teacher": TEACHER_NAME,
    "scale_counts": SCALE_COUNTS,
    "promoted": bool(promote),
    "val_util_mlp_p1s1_fase4": u_f4,
    "val_util_mlp_p1s1_fase3": u_f3,
    "val_util_heuristic": u_heu,
    "bc_linear_acc": {"train": acc_tr, "val": acc_va},
    "transitions": reports,
    "val_p1s1": pretty(rows_val_p1).to_dict(orient="records"),
    "test_p1s1": pretty(rows_test_p1).to_dict(orient="records"),
    "val_p3s2_subset": pretty(rows_val_p3).to_dict(orient="records"),
    "smoke_holdout": smoke_rows,
    "production": {
        "model_path": str(prod_mlp),
        "packing_mode": "online",
        "lookahead_p": 1,
        "select_s": 1,
    },
}
write_report(REPORTS_DIR / "04_escala.json", report)
print("\n=== model_path de producción ===")
print(f'parameters.model_path = "{prod_mlp}"')
print("packing_mode=online, lookahead_p=1, select_s=1")
print("promoted=", promote, "reporte", REPORTS_DIR / "04_escala.json")


=== model_path de producción ===
parameters.model_path = "/home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1.pt"
packing_mode=online, lookahead_p=1, select_s=1
promoted= False reporte /home/edmundo/packing-services/online_policy_ml/artifacts/reports/04_escala.json


Los 5 demo no se usaron para elegir si promover. Solo smoke de `is_valid`.